# 06 - Verify the complete exact-v3 reproduction

Reads only outputs from the current new run folder, asserts the frozen v3 targets, and writes a compact reproduction summary. It never accesses TEST.

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')

from pathlib import Path
import os, sys, json, shutil, subprocess, base64

DAIC = Path('/content/drive/MyDrive/DAIC_WOZ')
EXP_ROOT = DAIC / 'experiments' / 'exact_v3_reproduction'
ACTIVE = EXP_ROOT / 'ACTIVE_RUN.txt'
assert ACTIVE.is_file(), 'Run notebooks/00_start_new_run.ipynb first.'
RUN = EXP_ROOT / ACTIVE.read_text().strip()
assert RUN.is_dir(), RUN

REPO = 'engrkhubabahmad/reliability-aware-depression-kd'
CORE_COMMIT = '29087f39a46ddf756b6a082ff7148cf90e336ca7'
CODE = Path('/content/reliability-aware-depression-kd-exact-v3')

token = userdata.get('GITHUB_TOKEN')
assert token, 'Add GITHUB_TOKEN to Colab Secrets.'
auth = base64.b64encode(f'x-access-token:{token}'.encode()).decode()
env = os.environ.copy()
env['GIT_CONFIG_COUNT']='1'
env['GIT_CONFIG_KEY_0']='http.https://github.com/.extraheader'
env['GIT_CONFIG_VALUE_0']=f'AUTHORIZATION: basic {auth}'

if CODE.exists(): shutil.rmtree(CODE)
CODE.mkdir(parents=True)
subprocess.run(['git','-C',str(CODE),'init'],check=True,env=env)
subprocess.run(['git','-C',str(CODE),'remote','add','origin',f'https://github.com/{REPO}.git'],check=True,env=env)
subprocess.run(['git','-C',str(CODE),'fetch','--depth','1','origin',CORE_COMMIT],check=True,env=env)
subprocess.run(['git','-C',str(CODE),'checkout','--detach','FETCH_HEAD'],check=True,env=env)
head=subprocess.check_output(['git','-C',str(CODE),'rev-parse','HEAD'],text=True).strip()
assert head==CORE_COMMIT,(head,CORE_COMMIT)
subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(CODE/'requirements.txt')],check=True)
print('RUN:',RUN)
print('Pinned core:',head)


In [ ]:
import pandas as pd
TEXT=RUN/'03_student_text'
NOKD=RUN/'04_no_kd'
KD=RUN/'05_standard_kd'
TTEACH=RUN/'01_text_teacher'
ATEACH=RUN/'02_audio_teacher'/'dev_audit'

text=json.loads((TEXT/'metrics.json').read_text())['dev34']
nokd=json.loads((NOKD/'metrics.json').read_text())['dev34']
kd=json.loads((KD/'metrics.json').read_text())['dev34']
tt=json.loads((TTEACH/'text_target_audit.json').read_text())['dev34']
at=json.loads((ATEACH/'audit.json').read_text())['local_dev34']

rows=[
 {'Model':'IDIAP text teacher','Macro-F1':tt['macro_f1'],'Depressed F1':None,'AUROC':tt['auroc']},
 {'Model':'USSD audio teacher','Macro-F1':at['macro_f1'],'Depressed F1':at['classification_report']['1']['f1-score'],'AUROC':at['auroc_soft_mean_probability']},
 {'Model':'v3 text branch','Macro-F1':text['macro_f1'],'Depressed F1':text['depressed_f1'],'AUROC':text['auroc']},
 {'Model':'No-KD multimodal','Macro-F1':nokd['macro_f1'],'Depressed F1':nokd['depressed_f1'],'AUROC':nokd['auroc']},
 {'Model':'Standard KD','Macro-F1':kd['macro_f1'],'Depressed F1':kd['depressed_f1'],'AUROC':kd['auroc']},
]
df=pd.DataFrame(rows); display(df.round(4))

assert round(float(tt['macro_f1']),10)==round(0.8418604651162791,10)
assert round(float(at['macro_f1']),4)==0.7875
assert (round(float(text['macro_f1']),4),round(float(text['depressed_f1']),4),round(float(text['auroc']),4))==(0.7850,0.7407,0.8063)
assert (round(float(nokd['macro_f1']),4),round(float(nokd['depressed_f1']),4),round(float(nokd['auroc']),4))==(0.7043,0.6087,0.7470)
assert (round(float(kd['macro_f1']),4),round(float(kd['depressed_f1']),4),round(float(kd['auroc']),4))==(0.7236,0.6667,0.7668)

summary={'status':'PASS','core_commit':CORE_COMMIT,'run':str(RUN),'test_accessed':False,'results':rows}
OUT=RUN/'06_verification'; OUT.mkdir(exist_ok=False)
(OUT/'reproduction_summary.json').write_text(json.dumps(summary,indent=2)+'\n')
df.to_csv(OUT/'reproduction_table.csv',index=False)
print('================================================')
print('PASS: exact v3 DEV-34 reproduction is complete')
print('TEST was not read or scored.')
print('================================================')